In [ ]:
import json
import math
from pathlib import Path
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
DOCS_DIR        = "../../data/parsed_json/"
CHROMA_DIR      = "../../data/vector_db_v2"
COLLECTION_NAME = "rfp_docs"
MAX_CHUNK_SIZE  = 1000
BATCH_SIZE      = 500

# 임베딩 모델 선택
# "text-embedding-3-small" : 베이스라인, OpenAI API
# "bge-m3"                 : 한국어 강점, 로컬 무료
EMBEDDING_MODEL = "bge-m3"

1. 유틸리티

In [ ]:
def clean(val):
    # NaN / None → 빈 문자열 (Chroma 메타데이터는 NaN·None 불가)
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [ ]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    # 청크 메타데이터(페이로드) 생성
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

2. 청킹

In [ ]:
def chunk_section(section: dict, max_chunk_size: int = MAX_CHUNK_SIZE) -> list[dict]:
    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            if len(current_text) + len(block["content"]) > max_chunk_size and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })
    return results


In [ ]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    # JSON 문서 1개 → (청크 텍스트 리스트, 메타데이터 리스트)
    texts, metas = [], []

    # extraction_warnings 있으면 경고 출력 후 계속 처리
    warnings = doc.get("qa", {}).get("extraction_warnings", [])
    if warnings:
        print(f"  [WARN] {doc['doc_id']} — extraction_warnings: {warnings}")

    meta = doc.get("metadata", {})
    summary = str(clean(meta.get("사업요약")))
    사업명 = str(clean(meta.get("사업명")))
    발주기관 = str(clean(meta.get("발주기관")))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            # 사업명·발주기관·요약을 앞에 붙여 retrieval 정확도 향상
            prefix = (
                f"[사업명] {사업명}\n"
                f"[발주기관] {발주기관}\n"
                f"[요약] {summary}\n\n"
            )
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

3. 임베딩 모델

In [ ]:
def load_embedding_model(name: str):
    if name == "text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "text-embedding-3-large":
        return OpenAIEmbeddings(model="text-embedding-3-large")
    elif name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "ko-sroberta-multitask":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "embed-multilingual-v3":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model="embed-multilingual-v3.0")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")


4. Vector DB / Retrieval

In [ ]:
def get_vectorstore(embedding_model=None) -> Chroma:
    # 저장된 Chroma DB 불러오기 (검색 전용)
    if embedding_model is None:
        embedding_model = load_embedding_model(EMBEDDING_MODEL)
    return Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=CHROMA_DIR,
    )

In [ ]:
def get_retriever(vectorstore: Chroma, search_type: str = "similarity", k: int = 5):

    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k, "fetch_k": k * 4, "lambda_mult": 0.5},
        )
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

In [ ]:
def save_vectorstore():
    embedding_model = load_embedding_model(EMBEDDING_MODEL)

    if Path(CHROMA_DIR).exists():
        vectorstore = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=CHROMA_DIR,
        )
        existing_count = vectorstore._collection.count()

        # 전체 청크 수 계산
        docs_dir = Path(DOCS_DIR)
        all_texts, all_metas = [], []
        for json_file in sorted(docs_dir.glob("*.json")):
            with open(json_file, encoding="utf-8") as f:
                doc = json.load(f)
            texts, metas = process_doc(doc)
            all_texts.extend(texts)
            all_metas.extend(metas)

        if existing_count >= len(all_texts):
            print(f"[SKIP] 완료된 DB 존재 ({existing_count}개 청크)")
            return
        else:
            print(f"[RESUME] {existing_count}/{len(all_texts)}개, 이어서 진행")
            start_from = existing_count
    else:
        docs_dir = Path(DOCS_DIR)
        all_texts, all_metas = [], []
        for json_file in sorted(docs_dir.glob("*.json")):
            with open(json_file, encoding="utf-8") as f:
                doc = json.load(f)
            texts, metas = process_doc(doc)
            all_texts.extend(texts)
            all_metas.extend(metas)

        vectorstore = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=CHROMA_DIR,
        )
        start_from = 0

    for start in range(start_from, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vectorstore.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"\nChroma 저장 완료 (총 {vectorstore._collection.count()}개 청크)")

5. 실행

In [ ]:
# save_vectorstore() 호출
save_vectorstore()

6. 검색 테스트

In [ ]:
vs = get_vectorstore()
retriever = get_retriever(vs, k=5)

query = "보안 요구사항이 있는 사업은?"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"[{i}] 사업명: {doc.metadata.get('사업명')}")
    print(f"     발주기관: {doc.metadata.get('발주기관')}")
    print(f"     섹션: {doc.metadata.get('header_path')}")
    print(f"     내용:\n{doc.page_content[:300]}")
    print()

In [ ]:
# 7. top k 검색 테스트

In [ ]:
vs = get_vectorstore()
sample = vs.get(limit=100)
print(repr(sample["documents"][99]))
print(len(sample["documents"][99]))

In [ ]:
print(repr(sample["documents"][1]))
print(len(sample["documents"][1]))

In [ ]:
print(repr(sample["documents"][2]))
print(len(sample["documents"][2]))